# Chapter 10: API Integration Patterns
## AI for Networking and Security Engineers — Volume 1

This notebook demonstrates all five integration patterns from Chapter 10:

1. **ReAct Pattern Demo** — Simulate the Thought → Action → Observation loop
2. **AINetmiko Simulation** — AI-powered SSH troubleshooting (simulated device)
3. **NAPALM Config Validator** — AI validates config changes with risk assessment
4. **FastAPI REST Demo** — Call the AI API endpoint programmatically
5. **Circuit Breaker** — Watch the breaker trip, wait, and recover
6. **Exponential Backoff** — See the retry timing in action
7. **MCP Tool Discovery** — Simulate MCP server tool listing

> **Note**: Replace `YOUR_API_KEY` with your real Anthropic API key to run live demos.
> Demos 1–3 use real Claude API calls. Demos 4–7 demonstrate patterns without network devices.

In [ ]:
# Install dependencies
!pip install anthropic fastapi uvicorn httpx -q
print('Dependencies installed.')

In [ ]:
import os

# Load API key — works in Google Colab and locally
try:
    from google.colab import userdata
    api_key = userdata.get('ANTHROPIC_API_KEY')
    if not api_key:
        raise ValueError("ANTHROPIC_API_KEY not found in Colab secrets")
except Exception:
    import getpass
    api_key = getpass.getpass("Enter your Anthropic API key: ")

os.environ['ANTHROPIC_API_KEY'] = api_key
print('API key configured.')

---
## Demo 1: ReAct Pattern — Thought → Action → Observation Loop

The ReAct pattern is the foundation of all AI automation. The model thinks, decides what to do, your code executes it, the model observes the result and thinks again.

In networking terms: this is like OSPF database exchange — the model discovers what it needs, requests it, gets the data, processes it, and converges on an answer.

In [ ]:
from anthropic import Anthropic
import json

client = Anthropic()

# Simulated tool — in production this would be a real Netmiko call
SIMULATED_DEVICE_DATA = {
    'show ip bgp summary': '''
BGP router identifier 10.0.0.1, local AS number 65001
BGP table version is 15, main routing table version 15

Neighbor        V    AS MsgRcvd MsgSent   TblVer  InQ OutQ Up/Down  State/PfxRcd
10.1.1.2        4 65002       0       5        0    0    0 00:02:11 Idle
10.1.1.6        4 65003     125     130       15    0    0 02:15:33 24
''',
    'show ip bgp neighbors 10.1.1.2': '''
BGP neighbor is 10.1.1.2,  remote AS 65002, external link
  BGP state = Idle
  TCP connection failed
  Last reset 00:02:15, due to TCP connection failed
''',
    'show ip access-lists': '''
Extended IP access list BLOCK-BGP
    10 deny   tcp any any eq 179 (15 matches)
    20 permit ip any any
''',
    'show interfaces GigabitEthernet0/0': '''
GigabitEthernet0/0 is up, line protocol is up
  Internet address is 10.1.1.1/30
  MTU 1500 bytes, BW 1000000 Kbit/sec
'''
}

def execute_tool(command: str) -> str:
    """Simulate executing a CLI command."""
    for key, output in SIMULATED_DEVICE_DATA.items():
        if key in command:
            return output.strip()
    return f"% Unknown command: {command}"


def react_troubleshoot(symptom: str, max_steps: int = 5) -> str:
    """
    ReAct loop: Thought → Action → Observation until final answer.
    """
    print(f"\n{'='*60}")
    print(f"SYMPTOM: {symptom}")
    print('='*60)

    messages = [{
        "role": "user",
        "content": f"""You are a Cisco IOS network engineer troubleshooting an issue.

Problem: {symptom}

You have access to one tool: execute_command(command)

Use this EXACT format for each step:
Thought: [your reasoning]
Action: execute_command("{"command": "show ip bgp summary"}")
Observation: [I will provide the output]

After enough observations, end with:
Final Answer: [root cause and fix]

Start troubleshooting."""
    }]

    for step in range(max_steps):
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=600,
            messages=messages
        )

        text = response.content[0].text
        messages.append({"role": "assistant", "content": text})

        print(f"\n--- Step {step + 1} ---")
        print(text)

        # Check if model gave final answer
        if 'Final Answer:' in text:
            print(f"\n✓ ReAct loop converged in {step + 1} steps")
            return text

        # Extract command from Action line
        import re
        cmd_match = re.search(r'execute_command\(.*?"command"\s*:\s*"([^"]+)"', text)
        if cmd_match:
            command = cmd_match.group(1)
            observation = execute_tool(command)
            print(f"\n[TOOL OUTPUT] {command}:")
            print(observation)

            messages.append({
                "role": "user",
                "content": f"Observation: {observation}\n\nContinue troubleshooting."
            })

    return "Max steps reached — manual review needed"


# Run it
result = react_troubleshoot("BGP neighbor 10.1.1.2 is in Idle state")
print("\n" + "="*60)
print("ReAct loop complete!")

### ReAct in Plain Language

The ReAct pattern is how AI agents **reason and act** in a loop:

```
1. THINK  → "The user says BGP is down. I should check the neighbor state."
2. ACT    → Run 'show ip bgp neighbor' on the device
3. OBSERVE → See the output: "BGP state = Active, reason = remote AS mismatch"
4. THINK  → "AS mismatch found. Let me check the config."
5. ACT    → Run 'show run | section bgp'
6. OBSERVE → See the config: "neighbor 10.1.1.2 remote-as 65099"
7. CONCLUDE → "The remote AS is 65099 but should be 65002. Fix the config."
```

> This is how an experienced engineer troubleshoots — but automated.
> The AI decides what command to run next based on what it learned from the previous output.

---
## Demo 2: AINetmiko — The Intelligent SSH Session

This demo simulates the `AINetmiko.intelligent_troubleshoot()` method.
In production, replace `SimulatedDevice` with a real Netmiko `ConnectHandler`.

Key design: **Haiku** decides which commands to run (cheap classification). **Sonnet** synthesizes the diagnosis (complex multi-step reasoning).

In [ ]:
import re
import json
import time
from anthropic import Anthropic

client = Anthropic()

DANGEROUS_COMMANDS = ['reload', 'write erase', 'format', 'delete flash:', 'erase startup']

def is_safe(cmd: str) -> bool:
    return not any(d in cmd.lower() for d in DANGEROUS_COMMANDS)


class SimulatedDevice:
    """Replaces Netmiko ConnectHandler for Colab demo."""
    def send_command(self, command: str) -> str:
        return SIMULATED_DEVICE_DATA.get(command, f"% No output for: {command}")


def ai_troubleshoot(symptom: str, device_type: str = "cisco_ios") -> dict:
    """
    Full AINetmiko.intelligent_troubleshoot() logic.

    Step 1: Haiku selects commands (cheap)
    Step 2: Execute each command
    Step 3: Sonnet synthesizes root cause
    """
    device = SimulatedDevice()
    total_cost = 0.0

    print(f"\n🔍 Troubleshooting: {symptom}")
    print("-" * 50)

    # Step 1: Haiku picks commands
    t0 = time.time()
    cmd_resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        temperature=0,
        messages=[{"role": "user", "content": f"""
For this {device_type} issue: "{symptom}"
List 3-4 diagnostic show commands (read-only only).
Return ONLY a JSON array: ["cmd1", "cmd2"]"""}]
    )
    haiku_tokens = cmd_resp.usage.input_tokens + cmd_resp.usage.output_tokens
    haiku_cost = (cmd_resp.usage.input_tokens / 1e6 * 1.0) + (cmd_resp.usage.output_tokens / 1e6 * 5.0)
    print(f"✓ Haiku selected commands ({haiku_tokens} tokens, ${haiku_cost:.4f})")

    cmd_text = cmd_resp.content[0].text
    m = re.search(r'\[.*?\]', cmd_text, re.DOTALL)
    commands = json.loads(m.group()) if m else ["show ip bgp summary"]
    commands = [c for c in commands if is_safe(c)]
    print(f"Commands to run: {commands}")

    # Step 2: Execute
    results = []
    for cmd in commands:
        output = device.send_command(cmd)
        results.append({"command": cmd, "output": output})
        print(f"  ✓ Executed: {cmd}")

    # Step 3: Sonnet diagnosis
    all_output = "\n\n".join([f"=== {r['command']} ===\n{r['output']}" for r in results])

    diag_resp = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        temperature=0,
        messages=[{"role": "user", "content": f"""
You are a senior network engineer. Diagnose this issue.

Symptom: {symptom}
Device: {device_type}

Command outputs:
{all_output}

Return ONLY JSON:
{{
  "root_cause": "precise one sentence",
  "explanation": "2-3 sentences",
  "fix_commands": ["exact CLI commands"],
  "confidence": "high|medium|low",
  "risk_level": "safe|caution|dangerous"
}}"""}]
    )
    sonnet_tokens = diag_resp.usage.input_tokens + diag_resp.usage.output_tokens
    sonnet_cost = (diag_resp.usage.input_tokens / 1e6 * 3.0) + (diag_resp.usage.output_tokens / 1e6 * 15.0)
    total_cost = haiku_cost + sonnet_cost

    diag_text = diag_resp.content[0].text
    m2 = re.search(r'\{.*\}', diag_text, re.DOTALL)
    diagnosis = json.loads(m2.group()) if m2 else {"root_cause": "Unknown"}

    elapsed = time.time() - t0
    print(f"\n✓ Sonnet diagnosis complete ({sonnet_tokens} tokens, ${sonnet_cost:.4f})")
    print(f"Total elapsed: {elapsed:.1f}s | Total cost: ${total_cost:.4f}")

    print("\n" + "="*50)
    print("DIAGNOSIS:")
    print(f"  Root Cause : {diagnosis.get('root_cause', 'N/A')}")
    print(f"  Confidence : {diagnosis.get('confidence', 'N/A')}")
    print(f"  Risk Level : {diagnosis.get('risk_level', 'N/A')}")
    print(f"  Explanation: {diagnosis.get('explanation', 'N/A')}")
    fix_cmds = diagnosis.get('fix_commands', [])
    if fix_cmds:
        print(f"  Fix Commands:")
        for cmd in fix_cmds:
            safe_icon = "✅" if is_safe(cmd) else "🚫"
            print(f"    {safe_icon} {cmd}")

    return diagnosis


# Test it
diag = ai_troubleshoot("BGP neighbor 10.1.1.2 is in Idle state — TCP connection failing")
print("\n✓ AINetmiko simulation complete!")

---
## Demo 3: AI Config Validator — HITL Change Control

This demo shows the NAPALM + AI validation pattern without a real device.
We submit a candidate config change and get a full risk assessment.

The key principle: **AI calculates risk, human approves**. This is your automated Change Advisory Board.

In [ ]:
from anthropic import Anthropic
import json, re

client = Anthropic()

CURRENT_CONFIG = """
hostname core-rtr-01
!
ip access-list extended PERMIT-BGP
 permit tcp any any eq 179
 permit tcp any eq 179 any
!
router bgp 65001
 bgp router-id 10.0.0.1
 neighbor 10.1.1.2 remote-as 65002
 neighbor 10.1.1.2 password Str0ngPass!
 neighbor 10.1.1.2 update-source Loopback0
!
interface Loopback0
 ip address 10.0.0.1 255.255.255.255
!
interface GigabitEthernet0/0
 description Link to AS65002
 ip address 10.1.1.1 255.255.255.252
 ip access-group PERMIT-BGP in
"""

# Candidate config with a dangerous change hidden inside
CANDIDATE_CONFIG = """
hostname core-rtr-01
!
ip access-list extended PERMIT-BGP
 permit tcp any any eq 179
 permit tcp any eq 179 any
!
no service password-encryption
!
router bgp 65001
 bgp router-id 10.0.0.1
 neighbor 10.1.1.2 remote-as 65002
 neighbor 10.1.1.2 update-source Loopback0
!
interface GigabitEthernet0/0
 description Updated by automation script
 ip address 10.1.1.1 255.255.255.252
 ip access-group PERMIT-BGP in
 shutdown
"""


def validate_config_change(current: str, candidate: str, vendor: str = "cisco_ios") -> dict:
    """
    AI validates a config change and returns a risk assessment.
    Uses Sonnet — complex reasoning task.
    """
    prompt = f"""You are a senior {vendor} network engineer reviewing a production config change.

CURRENT RUNNING CONFIG:
```
{current}
```

CANDIDATE CONFIG (what will replace it):
```
{candidate}
```

Analyze all differences. Find:
1. What was REMOVED (could break things)
2. What was ADDED (new functionality or risk)
3. What was CHANGED (modifications)

Return ONLY valid JSON:
{{
  "safe_to_apply": true/false,
  "risk_level": "low|medium|high|critical",
  "issues": [
    {{"severity": "critical|high|medium|low", "description": "...", "line": "..."}}
  ],
  "good_changes": ["list of benign improvements"],
  "recommendation": "one sentence summary for the change window"
}}"""

    resp = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1500,
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    text = resp.content[0].text
    m = re.search(r'\{.*\}', text, re.DOTALL)
    return json.loads(m.group()) if m else {"safe_to_apply": False, "risk_level": "unknown"}


def print_validation_report(v: dict):
    risk_icons = {"low": "🟢", "medium": "🟡", "high": "🟠", "critical": "🔴", "unknown": "❓"}
    risk = v.get('risk_level', 'unknown')
    icon = risk_icons.get(risk, "❓")

    print("\n" + "="*55)
    print("AI CONFIG VALIDATION REPORT")
    print("="*55)
    print(f"Risk Level    : {icon} {risk.upper()}")
    print(f"Safe to Apply : {'✅ YES' if v.get('safe_to_apply') else '🚫 NO'}")
    print(f"Recommendation: {v.get('recommendation', 'N/A')}")

    issues = v.get('issues', [])
    if issues:
        print(f"\nIssues Found ({len(issues)}):")
        for issue in sorted(issues, key=lambda x: ['critical','high','medium','low'].index(x.get('severity','low'))):
            sev = issue.get('severity', 'unknown')
            icon2 = {"critical": "🔴", "high": "🟠", "medium": "🟡", "low": "🟢"}.get(sev, "⚪")
            print(f"  {icon2} [{sev.upper()}] {issue.get('description', '')}")
            if issue.get('line'):
                print(f"       Line: {issue['line']}")

    good = v.get('good_changes', [])
    if good:
        print(f"\nBenign Changes:")
        for g in good:
            print(f"  ✅ {g}")

    print("="*55)
    if not v.get('safe_to_apply'):
        print("🛑 BLOCKED: Fix critical/high issues before applying.")
    else:
        print("⏳ AWAITING HUMAN APPROVAL — review above before proceeding.")


# Run the validation
validation = validate_config_change(CURRENT_CONFIG, CANDIDATE_CONFIG)
print_validation_report(validation)
print("\n✓ Config validation demo complete!")

---
## Demo 4: REST API Client — Calling Your Team's AI Service

Once your FastAPI is deployed, any team member can call it. This demo simulates calling the REST API from another script.

In this demo we use `httpx` to simulate an HTTP call without needing a running server.
The `mock_api_call()` function replicates what the FastAPI endpoint would do.

In [ ]:
from anthropic import Anthropic
import json, re, time

client = Anthropic()

# --- Simulated REST API handler (replicates FastAPI endpoint logic) ---

def api_analyze_config(config: str, vendor: str = "cisco_ios", focus: str = None) -> dict:
    """Simulates POST /api/v1/analyze-config"""
    start = time.time()
    focus_clause = f"Pay special attention to: {focus}" if focus else ""

    prompt = f"""Analyze this {vendor} config for issues. {focus_clause}

Config:
```
{config[:4000]}
```

Return ONLY JSON:
{{
  "risk_level": "low|medium|high|critical",
  "findings": [
    {{"severity": "critical|high|medium|low", "issue": "...", "line": "...", "recommendation": "..."}}
  ],
  "summary": "overall paragraph assessment"
}}"""

    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1500,
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )

    text = resp.content[0].text
    m = re.search(r'\{.*\}', text, re.DOTALL)
    result = json.loads(m.group()) if m else {"risk_level": "unknown", "findings": [], "summary": "Parse error"}
    result['processing_time_ms'] = int((time.time() - start) * 1000)
    return result


# Example: NOC engineer calls the API
test_configs = [
    {
        "name": "Core Router — Security Audit",
        "config": """
hostname core-rtr-01
enable password cisco123
!
line vty 0 4
 transport input telnet
 no login
!
snmp-server community public RO
snmp-server community private RW
!
interface GigabitEthernet0/0
 ip address 10.1.1.1 255.255.255.252
 no shutdown
""",
        "focus": "security vulnerabilities"
    },
    {
        "name": "Distribution Switch — BGP Config",
        "config": """
router bgp 65001
 bgp log-neighbor-changes
 neighbor 10.1.1.2 remote-as 65002
 neighbor 10.1.1.2 password Str0ngP@ss!
 neighbor 10.1.1.2 soft-reconfiguration inbound
 neighbor 10.1.1.2 route-map ACCEPT-FROM-65002 in
 neighbor 10.1.1.2 route-map SEND-TO-65002 out
 maximum-paths 4
""",
        "focus": "bgp security and best practices"
    }
]

for test in test_configs:
    print(f"\n{'='*55}")
    print(f"Analyzing: {test['name']}")
    print(f"Focus: {test.get('focus', 'general')}")
    print('='*55)

    result = api_analyze_config(test['config'], focus=test.get('focus'))

    print(f"Risk Level      : {result.get('risk_level', 'N/A').upper()}")
    print(f"Processing Time : {result.get('processing_time_ms', 0)}ms")
    print(f"Summary         : {result.get('summary', 'N/A')[:200]}...")

    findings = result.get('findings', [])
    print(f"Findings ({len(findings)}):")
    for f_item in findings:
        sev = f_item.get('severity', 'info')
        icon = {"critical": "🔴", "high": "🟠", "medium": "🟡", "low": "🟢", "info": "🔵"}.get(sev, "⚪")
        print(f"  {icon} {f_item.get('issue', '')}")
        if f_item.get('recommendation'):
            print(f"     Fix: {f_item['recommendation']}")

print("\n✓ REST API demo complete!")

---
## Demo 5: Circuit Breaker — Handling AI API Failures

The Circuit Breaker prevents your automation from hammering a degraded API.

Watch the state transitions:
- `CLOSED` → normal operation
- `OPEN` → too many failures, stop calling (return fallback)
- `HALF_OPEN` → testing recovery with single request
- `CLOSED` → fully recovered

This demo simulates API failures without needing real API calls.

In [ ]:
import time
import random
from dataclasses import dataclass, field
from typing import Optional, Callable, Any

@dataclass
class CircuitBreakerConfig:
    failure_threshold: int = 3
    recovery_timeout: float = 5.0  # Short for demo
    success_threshold: int = 2

class CircuitBreakerState:
    CLOSED = "CLOSED"
    OPEN   = "OPEN"
    HALF_OPEN = "HALF-OPEN"

class CircuitBreaker:
    def __init__(self, config: CircuitBreakerConfig = None):
        self.config = config or CircuitBreakerConfig()
        self.state = CircuitBreakerState.CLOSED
        self.failure_count = 0
        self.success_count = 0
        self.last_failure_time: Optional[float] = None
        self.state_history = []

    def _log_transition(self, new_state: str):
        old = self.state
        self.state = new_state
        self.state_history.append((time.time(), old, new_state))
        print(f"  ⚡ Circuit Breaker: {old} → {new_state}")

    def call(self, fn: Callable, *args, fallback: Any = None, **kwargs) -> Any:
        if self.state == CircuitBreakerState.OPEN:
            elapsed = time.time() - (self.last_failure_time or 0)
            if elapsed >= self.config.recovery_timeout:
                self._log_transition(CircuitBreakerState.HALF_OPEN)
                self.success_count = 0
            else:
                remaining = self.config.recovery_timeout - elapsed
                print(f"  🔴 Circuit OPEN — {remaining:.1f}s remaining. Using fallback.")
                return fallback

        try:
            result = fn(*args, **kwargs)

            if self.state == CircuitBreakerState.HALF_OPEN:
                self.success_count += 1
                print(f"  ✓ Recovery test success ({self.success_count}/{self.config.success_threshold})")
                if self.success_count >= self.config.success_threshold:
                    self._log_transition(CircuitBreakerState.CLOSED)
                    self.failure_count = 0

            return result

        except Exception as e:
            self.failure_count += 1
            self.last_failure_time = time.time()
            print(f"  ✗ Failure {self.failure_count}/{self.config.failure_threshold}: {e}")

            if self.failure_count >= self.config.failure_threshold:
                self._log_transition(CircuitBreakerState.OPEN)

            if fallback is not None:
                return fallback
            raise


# Simulation
call_count = 0
fail_until = 7  # First 7 calls fail

def simulated_api_call(query: str) -> str:
    global call_count
    call_count += 1
    if call_count <= fail_until:
        raise ConnectionError(f"API unavailable (simulated failure #{call_count})")
    return f"AI analysis result for: {query}"


cb = CircuitBreaker(CircuitBreakerConfig(failure_threshold=3, recovery_timeout=3.0))
fallback_msg = {"status": "degraded", "message": "AI unavailable — manual review needed"}

queries = [
    "analyze BGP config",
    "check OSPF adjacencies",
    "validate ACL change",
    "diagnose interface flap",   # <- circuit should OPEN here
    "check SNMP community",      # <- OPEN, returns fallback
    "show route table",          # <- OPEN, returns fallback
    "analyze prefix list",       # <- OPEN, returns fallback
]

print("Circuit Breaker Demo")
print("=" * 50)
print(f"Settings: trip after {cb.config.failure_threshold} failures, "
      f"recover after {cb.config.recovery_timeout}s\n")

for i, query in enumerate(queries):
    print(f"\nCall {i+1}: '{query}'")
    result = cb.call(simulated_api_call, query, fallback=fallback_msg)
    print(f"  Result: {str(result)[:60]}")
    print(f"  State : [{cb.state}]")

# Simulate recovery — wait for timeout, then try again
print(f"\n⏳ Waiting {cb.config.recovery_timeout}s for circuit recovery...")
time.sleep(cb.config.recovery_timeout + 0.5)
fail_until = 0  # Now API works

print("\nRecovery test calls:")
for i in range(3):
    print(f"\nRecovery call {i+1}: 'test query {i+1}'")
    result = cb.call(simulated_api_call, f"recovery test {i+1}", fallback=fallback_msg)
    print(f"  Result: {result}")
    print(f"  State : [{cb.state}]")

print("\n✓ Circuit Breaker demo complete!")
print(f"State history: {[f'{s[1]}→{s[2]}' for s in cb.state_history]}")

---
## Demo 6: Exponential Backoff — The Right Way to Retry

When an API returns 429 (rate limited) or 500 (server error), retrying immediately makes things worse.

Exponential backoff waits: 1s, 2s, 4s, 8s — doubling each time, plus random jitter to prevent thundering herd.

Same principle as TCP binary exponential backoff on collision — designed to prevent multiple senders from retrying at exactly the same millisecond.

In [ ]:
import time
import random


# Custom error classes — defined first so exponential_backoff_call can catch them
class RateLimitError(Exception): pass
class ServerError(Exception):
    def __init__(self, status_code): super().__init__(f"HTTP {status_code}"); self.status_code = status_code
class ClientError(Exception):
    def __init__(self, status_code): super().__init__(f"HTTP {status_code}"); self.status_code = status_code


def exponential_backoff_call(fn, max_retries: int = 4, base_delay: float = 1.0, verbose: bool = True):
    """
    Retry fn() with exponential backoff + jitter.

    Wait pattern: 1s, 2s, 4s, 8s (+ random jitter 0-1s)
    Stops on 4xx (client errors) — only retries on 429/5xx.
    """
    for attempt in range(max_retries):
        try:
            if verbose:
                print(f"  Attempt {attempt + 1}/{max_retries}...")
            result = fn()
            if verbose:
                print(f"  ✓ Success on attempt {attempt + 1}")
            return result

        except RateLimitError:
            jitter = random.random()
            wait = (2 ** attempt) * base_delay + jitter
            if verbose:
                print(f"  ⚠️  429 Rate Limited. Backing off {wait:.1f}s...")
            time.sleep(wait)

        except ServerError as e:
            jitter = random.random()
            wait = (2 ** attempt) * base_delay + jitter
            if verbose:
                print(f"  ⚠️  {e.status_code} Server Error. Backing off {wait:.1f}s...")
            time.sleep(wait)

        except ClientError as e:
            if verbose:
                print(f"  ✗ {e.status_code} Client Error — not retrying (fix the request).")
            raise  # Don't retry client errors

    raise RuntimeError(f"All {max_retries} retries exhausted")


# --- Scenario 1: Rate limited, then succeeds ---
call_count = 0

def simulated_rate_limited_api():
    global call_count
    call_count += 1
    if call_count <= 2:
        raise RateLimitError("Rate limit exceeded")
    return {"result": "success", "attempts": call_count}


print("Scenario 1: Rate Limited (429) — recovers after 2 failures")
print("-" * 50)
t0 = time.time()
result = exponential_backoff_call(simulated_rate_limited_api, max_retries=4, base_delay=0.5)
elapsed = time.time() - t0
print(f"  Final result: {result}")
print(f"  Total time with backoff: {elapsed:.2f}s")

# --- Scenario 2: Client error (404) — should NOT retry ---
print("\nScenario 2: Client Error (404) — should NOT retry")
print("-" * 50)

def simulated_client_error_api():
    raise ClientError(404)

try:
    result = exponential_backoff_call(simulated_client_error_api)
except ClientError as e:
    print(f"  ✓ Correctly raised without retry: {e}")

# --- Visualize backoff timing ---
print("\nBackoff timing visualization:")
print("-" * 50)
print(f"{'Attempt':>8} | {'Wait (s)':>10} | {'Total elapsed':>15}")
print("-" * 40)
total = 0
for i in range(5):
    wait = (2 ** i) * 1.0
    total += wait
    bar = "█" * int(wait)
    print(f"{i+1:>8} | {wait:>10.1f}s | {total:>13.1f}s | {bar}")

print("\n✓ Exponential backoff demo complete!")

---
## Demo 7: MCP Tool Discovery — The Future of AI Integration

MCP (Model Context Protocol) is the emerging standard that replaces custom point-to-point integrations.

Instead of building a custom Python wrapper for every tool, you:
1. Build an MCP server that exposes tools
2. Any MCP-compatible AI client discovers them automatically

This demo simulates the MCP discovery protocol — what a client sees when it queries an MCP server.

In [ ]:
import json

# Simulated MCP server response — what tools does it expose?
NETWORK_MCP_SERVER_TOOLS = [
    {
        "name": "ping_host",
        "description": "Ping a network host and return reachability and RTT statistics",
        "inputSchema": {
            "type": "object",
            "properties": {
                "host": {"type": "string", "description": "IP address or hostname to ping"},
                "count": {"type": "integer", "description": "Number of ICMP echo requests", "default": 4}
            },
            "required": ["host"]
        }
    },
    {
        "name": "get_device_config",
        "description": "Retrieve running configuration from a network device via Netmiko",
        "inputSchema": {
            "type": "object",
            "properties": {
                "hostname": {"type": "string"},
                "device_type": {"type": "string", "description": "cisco_ios|juniper_junos|arista_eos"},
                "section": {"type": "string", "description": "Config section filter (bgp|ospf|interfaces|all)", "default": "all"}
            },
            "required": ["hostname", "device_type"]
        }
    },
    {
        "name": "validate_config",
        "description": "AI-powered config validation — returns risk level and issues",
        "inputSchema": {
            "type": "object",
            "properties": {
                "config": {"type": "string", "description": "Configuration text to validate"},
                "vendor": {"type": "string", "description": "Vendor type", "default": "cisco_ios"}
            },
            "required": ["config"]
        }
    },
    {
        "name": "troubleshoot_bgp",
        "description": "Full BGP troubleshooting workflow — runs diagnostic commands, returns root cause",
        "inputSchema": {
            "type": "object",
            "properties": {
                "hostname": {"type": "string"},
                "neighbor_ip": {"type": "string", "description": "BGP neighbor IP to troubleshoot"},
                "local_asn": {"type": "integer"}
            },
            "required": ["hostname", "neighbor_ip"]
        }
    },
    {
        "name": "search_knowledge_base",
        "description": "Semantic search over network documentation KB built in Chapter 9",
        "inputSchema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Natural language search query"},
                "n_results": {"type": "integer", "default": 5}
            },
            "required": ["query"]
        }
    }
]


def simulate_mcp_discovery():
    """Simulate what an MCP client sees when it queries the network MCP server."""
    print("MCP Tool Discovery")
    print("=" * 55)
    print("Querying network-tools MCP server...\n")

    print(f"Found {len(NETWORK_MCP_SERVER_TOOLS)} tools:\n")
    for i, tool in enumerate(NETWORK_MCP_SERVER_TOOLS, 1):
        print(f"{i}. {tool['name']}")
        print(f"   Description: {tool['description']}")
        required = tool['inputSchema'].get('required', [])
        props = tool['inputSchema'].get('properties', {})
        all_params = list(props.keys())
        optional_params = [p for p in all_params if p not in required]
        print(f"   Required params : {required}")
        if optional_params:
            print(f"   Optional params : {optional_params}")
        print()


def simulate_mcp_call(tool_name: str, arguments: dict) -> dict:
    """Simulate calling a tool on the MCP server."""
    print(f"\n→ MCP tool call: {tool_name}")
    print(f"  Arguments: {json.dumps(arguments, indent=2)}")

    # Simulated responses
    responses = {
        "ping_host": {"reachable": True, "avg_rtt_ms": 12.3, "packets_sent": 4, "packets_received": 4},
        "validate_config": {"risk_level": "medium", "issues": [{"severity": "high", "issue": "Telnet enabled"}]},
        "troubleshoot_bgp": {"root_cause": "ACL blocking TCP 179", "fix": "Remove deny tcp any any eq 179 from BLOCK-BGP"},
        "search_knowledge_base": {"results": [{"text": "OSPF area 0 is the backbone...", "score": 0.89}]}
    }

    result = responses.get(tool_name, {"status": "tool executed", "tool": tool_name})
    print(f"  Response: {json.dumps(result, indent=2)}")
    return result


# Demo
simulate_mcp_discovery()

print("\n" + "="*55)
print("Simulating AI agent calling MCP tools:")
print("="*55 + "\n")

# Simulate what happens when Claude decides to use these tools
simulate_mcp_call("ping_host", {"host": "10.1.1.2", "count": 4})
simulate_mcp_call("troubleshoot_bgp", {"hostname": "core-rtr-01", "neighbor_ip": "10.1.1.2", "local_asn": 65001})
simulate_mcp_call("search_knowledge_base", {"query": "What is the recommended OSPF area design for multi-site?", "n_results": 3})

print("\n" + "="*55)
print("MCP Comparison:")
print("  Traditional: Claude → custom Netmiko code")
print("               Claude → custom NAPALM code")
print("               Claude → custom REST API")
print("  MCP:         Claude → MCP Client → any MCP Server")
print("               Add new tool = add new MCP Server")
print("               Zero changes to Claude integration")
print("\n✓ MCP discovery demo complete!")

---
## Summary

| Pattern | When to Use | Key Class/Tool | Model |
|---------|-------------|----------------|-------|
| ReAct Loop | Troubleshooting with live device data | `react_troubleshoot()` | Haiku |
| AINetmiko | AI-enhanced SSH sessions | `ai_troubleshoot()` | Haiku + Sonnet |
| Config Validator | Pre-change risk assessment | `validate_config_change()` | Sonnet |
| REST API | Team-wide AI access | FastAPI endpoints | Haiku + Sonnet |
| Circuit Breaker | Production resilience | `CircuitBreaker` | N/A |
| Exponential Backoff | Retry on 429/5xx | `exponential_backoff_call()` | N/A |
| MCP | Standardized tool discovery | MCP Server | Any |

**Key rule**: AI proposes, human approves. Always. Especially for config changes.

**Next**: Chapter 11 — Testing and Validation. How do you test AI systems that produce non-deterministic outputs?